# A. Build search databases (run once, then reuse in notebooks B and C)

This notebook builds **standalone MMseqs2 databases with taxonomy** and packages them. Notebooks **B** (query → search → clusters → tree) and **C** (cluster growth) take one of these databases by folder or `.tar` package, so the slow work here is not repeated.

## What it can build
1. **A taxonomic subset of UniRef** (for example UniRef50 restricted to fungi). It downloads the UniRef FASTA file, keeps the entries of the chosen taxa (including all their descendants in the NCBI taxonomy) while reading the file once, and converts only those into a database. The full UniRef database is never built.
2. **A database from a FASTA file**, for example UniProt reference proteomes of fungi (complete proteomes, one entry per gene), or any FASTA whose headers carry `OX=taxid` or `TaxID=taxid`.

Each database ends up in its own folder `LIBRARY_DIR/<name>/` holding real files only (no links), plus `db_info.json`, which records what it is and lets B and C recognise it. A self-test searches one of its own sequences and checks that taxonomy comes back. Optionally the folder is packed into one `.tar` file.

## Where it runs
- **Google Colab (Linux):** works as before.
- **Local Linux or macOS:** works; MMseqs2 is installed for you or taken from the PATH.
- **Windows:** see "Windows" below. The Python code here uses no shell and no Linux paths, but **MMseqs2 itself is the weak point on native Windows**.

## Windows (Anaconda)
MMseqs2's own documentation describes Windows support only as a beta (built with Cygwin), the Windows download in its release notes carries warnings (slow multithreading, crashes with AVX2 on virtual machines), and the current release page may not list a Windows file at all. I could not test it. Two routes, in order of how much I would trust them:
1. **WSL2 (recommended).** WSL gives you a real Linux inside Windows. You install Miniconda *inside* WSL, create the environment there, start Jupyter there and open it in your normal Windows browser. Everything then behaves like Linux (and like Colab). Steps are in the box below.
2. **Native Windows.** Install MMseqs2 for Windows yourself, put `mmseqs.exe` (or `mmseqs.bat`) somewhere, and give its full path in `MMSEQS_EXE`. Steps that only need Python (download, taxon selection, packaging) will work; the steps that call MMseqs2 workflows (`createtaxdb`, the self-test search) are where it may fail. If it fails, use route 1.

**Environment and WSL setup (do once).**
1. Open **PowerShell as administrator** and run `wsl --install` (installs Ubuntu; restart Windows when asked; at first start choose a Linux user name and password). Check with `wsl -l -v` that the version is 2.
2. Open the **Ubuntu** app. Install Miniconda: `wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh` then `bash Miniconda3-latest-Linux-x86_64.sh`, answer yes, close and reopen Ubuntu.
3. Create the environment: `conda create -n seqwork -c conda-forge -c bioconda python=3.11 mmseqs2 mafft fasttree iqtree biopython pandas numpy matplotlib jupyterlab` and answer `y`.
4. Use it: `conda activate seqwork`, then `jupyter lab`. Copy the `http://localhost:8888/...` link it prints into your Windows browser. In Jupyter, upload this notebook (or open it from `/mnt/c/Users/<you>/...`, which is your Windows disk; for speed keep databases in the Linux home folder `~`, not under `/mnt/c`).
5. Give WSL enough memory if needed: create `C:\Users\<you>\.wslconfig` with `[wsl2]` and a line `memory=24GB`, then run `wsl --shutdown` in PowerShell.

Without WSL, in the Anaconda Prompt on Windows: `conda create -n seqwork -c conda-forge python=3.11 biopython pandas numpy matplotlib jupyterlab`, `conda activate seqwork`, `jupyter lab`. MMseqs2 and MAFFT must then be installed separately (the conda `bioconda` channel generally has no Windows builds).

## Steps, cost and output
| Step | What happens | Typical cost | Output |
|---|---|---|---|
| A0 Environment | Finds or installs MMseqs2 | Under a minute (not measured) | Version, free disk |
| A1 UniRef subset | Downloads the NCBI taxonomy and the UniRef FASTA (once), selects taxa while reading it, converts the selection | Download: 1.5 min for UniRef50 (8.4 GB) on Colab, depends on your line elsewhere. Reading 38.8 M entries in Python: not measured, I expect on the order of 15 to 40 minutes. Conversion of the selected few million sequences: about a million per minute in the earlier Colab run | Folder with the database and `db_info.json` |
| A2 FASTA database | Download or read a FASTA, convert, add taxonomy | Scales with the number of sequences | Same |
| A3 Check and package | Counts, self-test, optional `.tar` | One search against the database (not measured) | A table of the library, and the `.tar` |

## Things to keep in mind
- The Colab disk is wiped when the session ends. Point `LIBRARY_DIR` at a mounted Google Drive folder, or set `ARCHIVE_DIR` to one, to keep the result. Building straight on Drive is slower than building locally and copying.
- The UniRef download is kept (`KEEP_DOWNLOAD`) so that another taxon selection does not need it again; it is about 8.4 GB for UniRef50. Without it the only disk use is the selected subset.
- In UniRef, each entry's taxonomy is that of its representative member, so a UniRef50 entry stands in for sequences from several species.
- Browser downloads of multi-GB files from Colab often fail. Prefer Drive.

In [ ]:
#@title Parameters { display-mode: "form" }
#@markdown ### Where databases go
#@markdown Empty = a default folder next to the notebook (under /content in Colab). On Windows use forward slashes or a raw path such as `C:/data/mmseqs_library`.
LIBRARY_DIR = "" #@param {type:"string"}
WORKDIR = "" #@param {type:"string"}
#@markdown ### Tools
#@markdown Leave empty to use MMseqs2 from the PATH or let the notebook install it (Linux and macOS). On Windows give the full path to `mmseqs.exe` or `mmseqs.bat`.
MMSEQS_EXE = "" #@param {type:"string"}
AUTO_INSTALL = True #@param {type:"boolean"}
#@markdown ### Database 1: a taxonomic subset of UniRef
#@markdown Tick the groups to keep. Fungi, Metazoa (animals), Viridiplantae (plants) and Bacteria are NCBI taxids 4751, 33208, 33090 and 2. Add others as comma-separated taxids. All descendants of a taxid are included.
BUILD_UNIREF_SUBSET = True #@param {type:"boolean"}
UNIREF_VERSION = "UniRef50" #@param ["UniRef50", "UniRef90"]
SUBSET_NAME = "uniref50_fungi" #@param {type:"string"}
TAX_FUNGI = True #@param {type:"boolean"}
TAX_METAZOA = False #@param {type:"boolean"}
TAX_PLANTS = False #@param {type:"boolean"}
TAX_BACTERIA = False #@param {type:"boolean"}
EXTRA_TAXIDS = "" #@param {type:"string"}
#@markdown `UNIREF_FASTA`: path to an already downloaded `uniref50.fasta.gz` (or `.fasta`); empty = download it. `KEEP_DOWNLOAD` keeps the downloaded file.
UNIREF_FASTA = "" #@param {type:"string"}
KEEP_DOWNLOAD = True #@param {type:"boolean"}
#@markdown ### Database 2: from a FASTA file
#@markdown `UniProt reference proteomes, fungi` and `UniProtKB, all fungi (large)` are downloaded from UniProt. `custom FASTA` takes a URL or a local path in `FASTA_URL_OR_PATH`; headers need `OX=` or `TaxID=`.
BUILD_FASTA_DB = False #@param {type:"boolean"}
FASTA_NAME = "fungi_refproteomes" #@param {type:"string"}
FASTA_SOURCE = "UniProt reference proteomes, fungi" #@param ["UniProt reference proteomes, fungi", "UniProtKB, all fungi (large)", "custom FASTA"]
FASTA_URL_OR_PATH = "" #@param {type:"string"}
#@markdown ### Check and package
SELF_TEST = True #@param {type:"boolean"}
PACKAGE_TAR = True #@param {type:"boolean"}
ARCHIVE_DIR = "" #@param {type:"string"}
DOWNLOAD_PACKAGE = False #@param {type:"boolean"}
OVERWRITE_EXISTING = False #@param {type:"boolean"}

In [ ]:
# Advanced (optional).
# Folder with an NCBI taxdump (names.dmp, nodes.dmp, merged.dmp). Leave empty to download it from NCBI.
NCBI_TAXDUMP = ""

## A0. Environment
**What happens.** Finds MMseqs2 (`MMSEQS_EXE`, then the PATH, then a copy downloaded from the official GitHub releases on Linux and macOS). Defines the helpers: a runner for programs (no shell involved, so it behaves the same on every system) that shows MMseqs2's progress bar about every 20 seconds, a resumable downloader, the database builder, and the check-and-package routine.

**Expect.** Under a minute (not measured). Output: the system, the MMseqs2 version, CPU threads and free disk. If MMseqs2 cannot be found or started you get a message that says what to do (on Windows: see the notes at the top).

In [ ]:
#@title A0. Environment and helpers { display-mode: "form" }
import os, sys, re, json, shutil, hashlib, subprocess, time, platform, gzip, datetime, tarfile, zipfile, urllib.parse, urllib.request, urllib.error
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SYSTEM = platform.system()
THREADS = os.cpu_count() or 2
_base = Path("/content") if IN_COLAB else Path.cwd()
WORK = Path(WORKDIR) if WORKDIR else _base / "work_dbbuild"
WORK.mkdir(parents=True, exist_ok=True)
TMP = WORK / "tmp"; TMP.mkdir(exist_ok=True)
LIB = Path(LIBRARY_DIR) if LIBRARY_DIR else _base / "mmseqs_library"
LIB.mkdir(parents=True, exist_ok=True)
BIN = WORK / "bin"
NB_VERSION = "A2"
PROGRESS_RE = re.compile(r"^\s*\[[=> ]*\]")

def run(args, quiet=False, tail=25, every=20):
    """Run a program (list of arguments, no shell). MMseqs2 progress-bar lines are shown at most once per `every` seconds."""
    args = [str(a) for a in args]
    if not quiet:
        print("$", " ".join(f'"{a}"' if " " in a else a for a in args)[:300])
    t0, last, lines = time.time(), 0.0, []
    try:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except OSError as e:
        raise RuntimeError(f"Could not start {args[0]!r}: {e}")
    for line in p.stdout:
        lines.append(line)
        if quiet:
            continue
        if PROGRESS_RE.match(line):
            if time.time() - last >= every:
                print(f"  [{time.time() - t0:6.0f}s] {line.strip()[:110]}"); last = time.time()
        else:
            print(line, end="")
    rc = p.wait()
    if rc:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(args)[:300]}\n" + "".join(lines[-tail:]))
    if not quiet:
        print(f"  (finished in {time.time() - t0:.0f}s)")
    return "".join(lines)

def _find(names, extra_dirs=()):
    for n in names:
        w = shutil.which(n)
        if w: return w
    for d in extra_dirs:
        if Path(d).exists():
            for n in names:
                for p in Path(d).rglob(n):
                    if p.is_file(): return str(p)
    return None

def find_mmseqs():
    if MMSEQS_EXE:
        if not Path(MMSEQS_EXE).exists(): raise FileNotFoundError(f"MMSEQS_EXE {MMSEQS_EXE!r} does not exist.")
        return MMSEQS_EXE
    return _find(["mmseqs", "mmseqs.bat", "mmseqs.exe"], [BIN])

def install_mmseqs():
    BIN.mkdir(parents=True, exist_ok=True)
    base = "https://github.com/soedinglab/MMseqs2/releases/latest/download/"
    if SYSTEM == "Linux":
        if platform.machine().lower() in ("aarch64", "arm64"):
            asset = "mmseqs-linux-arm64.tar.gz"
        else:
            try: flags = open("/proc/cpuinfo").read()
            except OSError: flags = ""
            asset = "mmseqs-linux-" + ("avx2" if " avx2" in flags else ("sse41" if " sse4_1" in flags else "sse2")) + ".tar.gz"
    elif SYSTEM == "Darwin":
        asset = "mmseqs-osx-universal.tar.gz"
    else:
        raise RuntimeError("There is no automatic MMseqs2 install for this system. On Windows use WSL2 (see the notes at the top of this notebook), "
                           "or install MMseqs2 yourself and set MMSEQS_EXE.")
    dest = BIN / asset
    print("Downloading", base + asset)
    urllib.request.urlretrieve(base + asset, dest)
    with tarfile.open(dest) as t: t.extractall(BIN)
    dest.unlink()

MMSEQS = find_mmseqs()
if not MMSEQS and AUTO_INSTALL:
    install_mmseqs(); MMSEQS = find_mmseqs()
if not MMSEQS:
    raise RuntimeError("MMseqs2 not found. Install it (conda: `conda install -c conda-forge -c bioconda mmseqs2` on Linux/macOS/WSL) and set MMSEQS_EXE if it is not on the PATH.")

def mm(*args, **kw):
    return run([MMSEQS, *args], **kw)

MMSEQS_VERSION = mm("version", quiet=True).strip().splitlines()[-1]

def download(url, dest, resume=False, every=20):
    """Download with a progress line every `every` seconds. With resume=True a partial file is continued if the server allows it."""
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    done = dest.stat().st_size if (resume and dest.exists()) else 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    try:
        r = urllib.request.urlopen(req, timeout=60)
    except urllib.error.HTTPError as e:
        if e.code == 416 and done:             # already complete
            return dest
        raise RuntimeError(f"Download failed ({e.code}) for {url}")
    total = r.headers.get("Content-Length"); total = int(total) + done if total else None
    if done and r.status != 206:
        done = 0                                # server ignored the range: start over
    mode = "ab" if done else "wb"
    t0 = last = time.time(); n = done
    with open(dest, mode) as f:
        while True:
            b = r.read(1 << 20)
            if not b: break
            f.write(b); n += len(b)
            if time.time() - last >= every:
                print(f"  {n / 1e9:.2f}" + (f" / {total / 1e9:.2f}" if total else "") + f" GB after {time.time() - t0:.0f}s"); last = time.time()
    if total and n != total:
        raise RuntimeError(f"Download incomplete: {n} of {total} bytes. Run the cell again to resume.")
    print(f"  downloaded {n / 1e9:.2f} GB in {time.time() - t0:.0f}s -> {dest}")
    return dest

def keyof(*objs):
    return hashlib.sha1(json.dumps(objs, sort_keys=True, default=str).encode()).hexdigest()[:12]

def n_seqs(db):
    return sum(1 for _ in open(str(db) + ".index"))

def dir_size_gb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e9

def ensure_taxdump():
    """Folder with names.dmp, nodes.dmp, merged.dmp: NCBI_TAXDUMP if given, else downloaded from NCBI once."""
    if NCBI_TAXDUMP:
        d = Path(NCBI_TAXDUMP)
        if not (d / "nodes.dmp").exists(): raise FileNotFoundError(f"No nodes.dmp in NCBI_TAXDUMP {d}")
        return d
    d = WORK / "taxdump"
    if (d / "nodes.dmp").exists() and (d / "names.dmp").exists():
        return d
    d.mkdir(parents=True, exist_ok=True)
    tgz = download("https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz", WORK / "taxdump.tar.gz")
    with tarfile.open(tgz) as t:
        for m in t.getmembers():
            if m.name in ("names.dmp", "nodes.dmp", "merged.dmp", "delnodes.dmp"):
                m.name = Path(m.name).name; t.extract(m, d)
    tgz.unlink()
    return d

def descendants(taxids, dump):
    """All taxids at or below the given ones (from nodes.dmp), plus old taxids merged into them (merged.dmp)."""
    children = {}
    with open(Path(dump) / "nodes.dmp") as f:
        for line in f:
            p = line.split("\t|\t")
            t, par = int(p[0]), int(p[1])
            if t != par: children.setdefault(par, []).append(t)
    out, stack = set(), list(taxids)
    while stack:
        t = stack.pop()
        if t in out: continue
        out.add(t); stack.extend(children.get(t, ()))
    mf = Path(dump) / "merged.dmp"
    if mf.exists():
        with open(mf) as f:
            for line in f:
                p = line.split("\t|\t"); old, new = int(p[0]), int(p[1].split("\t")[0])
                if new in out: out.add(old)
    return out

def build_db_from_fasta(fa, db, workdir):
    """FASTA (plain or gzip) -> MMseqs2 database with taxonomy. Headers must carry OX=taxid or TaxID=taxid."""
    db, workdir = Path(db), Path(workdir)
    mm("createdb", fa, db, "-v", 2)
    with open(fa, "rb") as fh:
        is_gz = fh.read(2) == b"\x1f\x8b"
    mapf = workdir / "taxmap.tsv"; n_h = n_miss = 0
    # createdb stores the sequence identifiers in db.lookup in input order (for sp|ACC|NAME headers it keeps only ACC),
    # so the mapping is written in lockstep with the headers, using the lookup's own identifier.
    with (gzip.open if is_gz else open)(fa, "rt") as fi, open(mapf, "w") as fo, open(f"{db}.lookup") as lk:
        for line in fi:
            if line.startswith(">"):
                n_h += 1
                ident = next(lk).split("\t")[1]
                m = re.search(r"(?:OX|TaxID)=(\d+)", line)
                if m:
                    fo.write(f"{ident}\t{m.group(1)}\n")
                else:
                    n_miss += 1
        if next(lk, None) is not None:
            raise RuntimeError("db.lookup has more entries than the FASTA has headers; cannot map taxonomy ids safely.")
    print(f"{n_h:,} sequences; {n_miss:,} headers without OX= or TaxID=")
    if n_h == 0 or n_miss > 0.5 * n_h:
        raise RuntimeError("Most headers carry no taxonomy id (OX= or TaxID=), so taxonomy cannot be built. Use a FASTA with UniProt-style or UniRef-style headers.")
    dump = ensure_taxdump()
    shutil.rmtree(workdir / "tmp_tax", ignore_errors=True)
    mm("createtaxdb", db, workdir / "tmp_tax", "--tax-mapping-file", mapf, "--ncbi-tax-dump", dump, "--threads", THREADS)
    return n_h

def first_sequence(db):
    with open(f"{db}.index") as f:
        key, off, ln = f.readline().split("\t")[:3]
    with open(db, "rb") as f:
        f.seek(int(off))
        return f.read(int(ln)).decode().replace("\x00", "").strip()

def finish_db(outdir, name, kind, source, taxids):
    """Check a finished database folder, write db_info.json and optionally pack it. Returns the info dict."""
    outdir = Path(outdir); db = outdir / "db"
    for ext in (".dbtype", ".index", ".lookup", "_h", "_h.index", "_mapping", "_taxonomy"):
        if not Path(str(db) + ext).exists():
            raise RuntimeError(f"{outdir} is missing db{ext}; the database is incomplete.")
    links = [p.name for p in outdir.iterdir() if p.is_symlink()]
    if links:
        raise RuntimeError(f"{outdir} contains links ({links[:3]}...), so it is not standalone.")
    n = n_seqs(db)
    with open(str(db) + "_mapping") as f:
        n_tax = len({l.split("\t")[1].strip() for l in f if "\t" in l})
    checks = {"standalone": True, "self_test": "not run"}
    if SELF_TEST:
        q = first_sequence(db)
        qf = WORK / "selftest.fa"; qf.write_text(f">selftest\n{q}\n")
        out = WORK / "selftest.tsv"
        shutil.rmtree(TMP / "selftest", ignore_errors=True)
        mm("easy-search", qf, db, out, TMP / "selftest", "-s", 4, "-e", "1e-10", "--max-seqs", 5, "--threads", THREADS, "-v", 1,
           "--format-output", "query,target,fident,taxid,taxlineage")
        rows = [l.rstrip("\n").split("\t") for l in open(out)]
        good = [r for r in rows if len(r) >= 5 and float(r[2]) >= 0.99 and r[3] not in ("", "0") and r[4]]
        if not good:
            raise RuntimeError("Self-test failed: searching a sequence of this database against itself did not return itself with a taxonomy lineage. "
                               f"First rows: {rows[:3]}")
        checks["self_test"] = f"passed (hit {good[0][1]}, taxid {good[0][3]})"
        print("Self-test:", checks["self_test"])
    created = datetime.datetime.now().isoformat(timespec="seconds")
    info = {"format": 1, "name": name, "kind": kind, "source": source, "taxids": taxids, "db_prefix": "db",
            "sequences": n, "distinct_taxids": n_tax, "created": created, "mmseqs_version": MMSEQS_VERSION,
            "notebook": NB_VERSION, "id": keyof(name, n, created), "checks": checks}
    (outdir / "db_info.json").write_text(json.dumps(info, indent=1))
    info["size_gb"] = round(dir_size_gb(outdir), 2)
    print(f"{name}: {n:,} sequences, {n_tax:,} distinct taxids, {info['size_gb']} GB in {outdir}")
    return info

def package(outdir, info):
    outdir = Path(outdir)
    dest = Path(ARCHIVE_DIR) if ARCHIVE_DIR else LIB
    dest.mkdir(parents=True, exist_ok=True)
    tar = dest / f"{outdir.name}.tar"
    if PACKAGE_TAR:
        with tarfile.open(tar, "w") as tf:
            tf.add(outdir, arcname=outdir.name)
        print(f"Packed {tar} ({tar.stat().st_size / 1e9:.2f} GB)")
        if DOWNLOAD_PACKAGE and IN_COLAB:
            from google.colab import files as colab_files
            colab_files.download(str(tar))
    return tar if PACKAGE_TAR else None

print(f"{SYSTEM} | MMseqs2 {MMSEQS_VERSION} ({MMSEQS})")
print("threads:", THREADS, "| free disk (GB) in library dir:", round(shutil.disk_usage(LIB).free / 1e9, 1))
print("library:", LIB, "| work:", WORK)
BUILT = []

## A1. Database 1: a taxonomic subset of UniRef
**What happens.**
1. The NCBI taxonomy (`taxdump.tar.gz`, about 60 MB) is downloaded once, and every taxid at or below your chosen taxa is collected.
2. The UniRef FASTA (`uniref50.fasta.gz`, 8.4 GB for UniRef50) is downloaded from UniProt, or taken from `UNIREF_FASTA`. A partly downloaded file is resumed if you run the cell again.
3. The file is read once, entry by entry. Entries whose `TaxID=` is in the collected set are written to a smaller FASTA. This reads the whole file in Python, so it takes a while, but it replaces the old route (converting all of UniRef to a database first, which took 37 minutes and 22 GB).
4. The selection is converted with `createdb`, taxonomy is added with `createtaxdb` using the same taxonomy, and the result is checked (sequence count against the selection, all files present, no links).

**Expect.** Download: 1.5 minutes on Colab for UniRef50, longer on a home connection. Reading and selecting 38.8 M entries: not measured; the progress line gives the rate, and I expect roughly 15 to 40 minutes. Conversion: about a million sequences per minute (earlier Colab run). Disk: the download plus a few GB for the subset.

**Parameters that matter.**
- `UNIREF_VERSION`: UniRef50 is the compact choice for remote homologs; UniRef90 is several times larger and gives denser species sampling.
- The `TAX_*` boxes and `EXTRA_TAXIDS`: which entries are kept. Several can be combined in one database; make separate ones by running the cell again with another `SUBSET_NAME`.
- `UNIREF_FASTA` and `KEEP_DOWNLOAD`: reuse or keep the big download.
- `SUBSET_NAME`: the folder name under `LIBRARY_DIR`. An existing folder is kept unless `OVERWRITE_EXISTING` is ticked.

In [ ]:
#@title A1. Build a UniRef subset { display-mode: "form" }
if not BUILD_UNIREF_SUBSET:
    print("BUILD_UNIREF_SUBSET is not ticked: skipping A1.")
else:
    PRESET_NAMES = {"fungi": 4751, "metazoa": 33208, "plants": 33090, "bacteria": 2}
    taxids = sorted({PRESET_NAMES[n] for n, on in [("fungi", TAX_FUNGI), ("metazoa", TAX_METAZOA), ("plants", TAX_PLANTS),
                                                   ("bacteria", TAX_BACTERIA)] if on} | {int(x) for x in re.findall(r"\d+", EXTRA_TAXIDS)})
    if not taxids:
        raise ValueError("No taxonomic group selected. Tick a group or give EXTRA_TAXIDS, or untick BUILD_UNIREF_SUBSET.")
    name = re.sub(r"\W+", "_", SUBSET_NAME).strip("_") or "subset"
    outdir = LIB / name
    if (outdir / "db.dbtype").exists() and not OVERWRITE_EXISTING:
        print(f"{outdir} already exists: keeping it (tick OVERWRITE_EXISTING to rebuild).")
        info = json.loads((outdir / "db_info.json").read_text()); BUILT.append((outdir, info))
    else:
        dump = ensure_taxdump()
        wanted = descendants(taxids, dump)
        print(f"Taxids {taxids}: {len(wanted):,} taxa at or below them.")
        wanted_b = {str(t).encode() for t in wanted}
        if UNIREF_FASTA:
            src = Path(UNIREF_FASTA)
            if not src.exists(): raise FileNotFoundError(f"UNIREF_FASTA {src} does not exist.")
        else:
            src = WORK / "downloads" / f"{UNIREF_VERSION.lower()}.fasta.gz"
            url = f"https://ftp.uniprot.org/pub/databases/uniprot/uniref/{UNIREF_VERSION.lower()}/{UNIREF_VERSION.lower()}.fasta.gz"
            need = {"UniRef50": 10, "UniRef90": 40}[UNIREF_VERSION]
            free = shutil.disk_usage(WORK).free / 1e9
            if free < need: raise RuntimeError(f"Only {free:.0f} GB free in {WORK}; the {UNIREF_VERSION} download needs about {need} GB.")
            print("Downloading", url)
            download(url, src, resume=True)
        selfa = WORK / f"{name}.fasta"
        tax_re = re.compile(rb"TaxID=(\d+)")
        opener = gzip.open if str(src).endswith(".gz") else open
        n_all = n_keep = 0; keep = False; t0 = last = time.time()
        print("Reading", src, "and selecting entries ...")
        with opener(src, "rb") as fi, open(selfa, "wb") as fo:
            for line in fi:
                if line[:1] == b">":
                    n_all += 1
                    m = tax_re.search(line)
                    keep = bool(m) and m.group(1) in wanted_b
                    if keep: n_keep += 1
                    if time.time() - last >= 20:
                        print(f"  {n_all:,} entries read, {n_keep:,} kept ({n_all / (time.time() - t0):,.0f} entries/s)"); last = time.time()
                if keep:
                    fo.write(line)
        print(f"Read {n_all:,} entries in {time.time() - t0:.0f}s; kept {n_keep:,}.")
        if n_keep == 0:
            raise RuntimeError(f"No entries of taxids {taxids} found in {src}. Check UNIREF_FASTA and the taxids.")
        if not KEEP_DOWNLOAD and not UNIREF_FASTA:
            src.unlink(); print("Deleted the download (KEEP_DOWNLOAD is off).")
        if outdir.exists():
            shutil.rmtree(outdir)
        outdir.mkdir(parents=True)
        build_db_from_fasta(selfa, outdir / "db", outdir)
        selfa.unlink(); (outdir / "taxmap.tsv").unlink(missing_ok=True); shutil.rmtree(outdir / "tmp_tax", ignore_errors=True)
        if n_seqs(outdir / "db") != n_keep:
            raise RuntimeError("The database has a different number of sequences from the selection.")
        info = finish_db(outdir, name, "uniref_subset", {"uniref": UNIREF_VERSION}, taxids)
        BUILT.append((outdir, info))

## A2. Database 2: from a FASTA file
**What happens.** Downloads (or copies) a FASTA, converts it with `mmseqs createdb`, writes a taxid mapping from the `OX=` or `TaxID=` field of each header, and adds taxonomy with `createtaxdb`. The default source is the UniProt reference proteomes of fungi. Because a proteome holds every protein of a species, this kind of database is the right one for asking which sequences a species has, and which it lacks; a clustered database like UniRef50 cannot say.

**Expect.** Not measured. The download depends on the size of the set and on UniProt; conversion runs at roughly a million sequences per minute in the earlier Colab run. If the download is empty or fails, the cell says so.

**Parameters that matter.**
- `FASTA_SOURCE`: reference proteomes are one curated set per species. All UniProtKB for fungi is larger and redundant. The UniProt queries used are `taxonomy_id:4751 AND keyword:KW-1185` (reference proteomes, which I believe is the right keyword but could not verify) and `taxonomy_id:4751`. Check them on the UniProt website if the result looks wrong.
- `custom FASTA` with `FASTA_URL_OR_PATH`: use any FASTA with taxonomy ids in the headers.
- `FASTA_NAME`: the folder name under `LIBRARY_DIR`.

In [ ]:
#@title A2. Build a database from a FASTA { display-mode: "form" }
if not BUILD_FASTA_DB:
    print("BUILD_FASTA_DB is not ticked: skipping A2.")
else:
    name = re.sub(r"\W+", "_", FASTA_NAME).strip("_") or "fasta_db"
    outdir = LIB / name
    if (outdir / "db.dbtype").exists() and not OVERWRITE_EXISTING:
        print(f"{outdir} already exists: keeping it (tick OVERWRITE_EXISTING to rebuild).")
        info = json.loads((outdir / "db_info.json").read_text()); BUILT.append((outdir, info))
    else:
        S2_QUERIES = {"UniProt reference proteomes, fungi": "(taxonomy_id:4751) AND (keyword:KW-1185)",
                      "UniProtKB, all fungi (large)": "(taxonomy_id:4751)"}
        if FASTA_SOURCE == "custom FASTA":
            if not FASTA_URL_OR_PATH:
                raise ValueError("FASTA_SOURCE is 'custom FASTA' but FASTA_URL_OR_PATH is empty.")
            src = FASTA_URL_OR_PATH
            source = {"fasta": src}
        else:
            qs = S2_QUERIES[FASTA_SOURCE]
            src = "https://rest.uniprot.org/uniprotkb/stream?compressed=true&format=fasta&query=" + urllib.parse.quote(qs)
            source = {"uniprot_query": qs}
        if outdir.exists():
            shutil.rmtree(outdir)
        outdir.mkdir(parents=True)
        fa = WORK / f"{name}.fasta.gz"
        if src.startswith("http"):
            download(src, fa)
        else:
            shutil.copy(src, fa)
        if fa.stat().st_size == 0:
            raise RuntimeError("The FASTA is empty. Check the UniProt query (see the text above this cell) or give FASTA_URL_OR_PATH.")
        build_db_from_fasta(fa, outdir / "db", outdir)
        (outdir / "taxmap.tsv").unlink(missing_ok=True); shutil.rmtree(outdir / "tmp_tax", ignore_errors=True)
        info = finish_db(outdir, name, "fasta", source, [])
        BUILT.append((outdir, info))

## A3. Check, package and summarise
**What happens.** Every database built or kept in this run was already checked when it was written: all files present, no links, sequence and taxid counts, and (if `SELF_TEST` is ticked) a search of its own first sequence that must return itself with a taxonomy lineage. This cell lists them, packs each folder into one `.tar` if `PACKAGE_TAR` is ticked, and tells you what to enter in notebooks B and C.

**Expect.** A table of the databases. Packing copies the data once, so it needs as much free disk as the database and takes minutes for a few GB (not measured). `ARCHIVE_DIR` set to a mounted Drive folder writes the `.tar` there. `DOWNLOAD_PACKAGE` starts a browser download (Colab only), which often fails for multi-GB files, so Drive is the safer route.

**Parameters that matter.** `PACKAGE_TAR`, `ARCHIVE_DIR`, `DOWNLOAD_PACKAGE`, `SELF_TEST`.

In [ ]:
#@title A3. Check, package and summarise { display-mode: "form" }
import pandas as pd
from IPython.display import display
if not BUILT:
    print("Nothing was built or kept in this run.")
else:
    rows = []
    for outdir, info in BUILT:
        tar = package(outdir, info)
        rows.append({"name": info["name"], "kind": info["kind"], "sequences": f"{info['sequences']:,}",
                     "distinct taxids": f"{info['distinct_taxids']:,}", "size (GB)": round(dir_size_gb(outdir), 2),
                     "self-test": info["checks"]["self_test"], "folder": str(outdir), "package": str(tar) if tar else ""})
    display(pd.DataFrame(rows))
    print("In notebooks B and C, set DATABASE to the folder (or the .tar) of the database you want to search.")